# V11 Cluster Naming Assignment

Assigns **persona / learning-intent names** (all clusters) and optional **demographic narrative labels** (subset) to every developer in `dev_lifecycle_cluster_membership_v11_final`.

**Outputs**
- `outputs/v11_cluster_naming/dev_v11_cluster_names.parquet`
- `outputs/v11_cluster_naming/v11_cluster_name_reference.csv`
- DuckDB table `dev_v11_cluster_names` (optional write-back)

Every cluster gets persona names. Demographic audience / archetype fields are populated only where the Demographics Team slides define them.

## 1. Setup

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

PROJECT_DIR = Path(".")
DB_PATH = PROJECT_DIR / "developer_project.duckdb"
MEMBERSHIP_TABLE = "dev_lifecycle_cluster_membership_v11_final"
MEMBERSHIP_PARQUET = PROJECT_DIR / "outputs/v11_cluster_exports/dev_lifecycle_cluster_membership_v11_final.parquet"
OUTPUT_DIR = PROJECT_DIR / "outputs/v11_cluster_naming"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WRITE_DUCKDB = True
DUCKDB_OUTPUT_TABLE = "dev_v11_cluster_names"

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

## 2. Cluster name registry

Persona names apply to **every** `cluster_key`. Demographic fields are optional overlays from the Demographics Team deck.

In [2]:
# Persona / learning-intent names (HDBSCAN cluster naming sheet)
PERSONA_NAMES = {
    # Active
    "active_0": ("GenAI Evaluation Cohort", "GenAI Evaluators with Education-Led Activity"),
    "active_noise": ("Active Multi-Path Users", "Mixed Active Users with Broad Activity Paths"),
    "active_1": ("Lightweight GenAI Evaluators", "Lower-Depth GenAI Evaluators"),
    "active_2": ("GenAI Community Participants", "GenAI Evaluators with Community-Led Activity"),
    "active_3": ("High-Intent GenAI Evaluators", "Higher-Intent GenAI Evaluators with Stronger Training Activity"),
    "active_4": ("Event-Driven GenAI Evaluators", "GenAI Evaluators with Webinar-Led Activity"),
    "active_5": ("Core CUDA Builders", "CUDA Builders with Strong API and Download Depth"),
    # Cooling
    "cooling_0": ("Cooling GenAI Education Users", "Cooling GenAI Learners with Education-Led Activity"),
    "cooling_noise": ("Cooling Multi-Path Users", "Mixed Cooling Users with Download-Led Activity"),
    "cooling_1": ("Cooling GenAI Training Users", "Cooling GenAI Learners with Training-Led Activity"),
    "cooling_2": ("Cooling Cross-Persona Learners", "Cooling Mixed-Persona Learners"),
    "cooling_3": ("Cooling CUDA Education Users", "Cooling CUDA Learners with Education-Led Activity"),
    "cooling_4": ("Cooling General Learning Users", "Cooling General Learners with Mixed Education Activity"),
    "cooling_5": ("Cooling CUDA Community Users", "Cooling CUDA Learners with Community-Led Activity"),
    "cooling_6": ("Cooling High-Intent GenAI Users", "Cooling Higher-Intent GenAI Learners"),
    # At-risk
    "at_risk_0": ("At-Risk Technical Learners", "At-Risk Mixed Technical Learners with Download and Training History"),
    "at_risk_1": ("At-Risk Builder-Learners", "At-Risk Mixed Learners with Builder Residue"),
    "at_risk_2": ("At-Risk GenAI Training Users", "At-Risk GenAI Learners with Training-Led History"),
    "at_risk_3": ("At-Risk GenAI Hands-On Learners", "At-Risk GenAI Learners with More Hands-On Technical Activity"),
    "at_risk_4": ("At-Risk CUDA Education Users", "At-Risk CUDA Learners with Education-Led History"),
    "at_risk_5": ("At-Risk Event-Led Learners", "At-Risk Learners with Webinar-Led History"),
    "at_risk_noise": ("At-Risk Multi-Path Users", "Mixed At-Risk Users with Broad Download-Led History"),
    # Dormant
    "Dormant_Low_Depth": ("Dormant Low-Depth Visitors", "Dormant Low-Depth Historical Users"),
    "Dormant_Former_Builders": ("Dormant Former Builders", "Dormant Historical Builders"),
    "Dormant_One_Time_Users": ("Dormant One-Time Users", "Dormant Single-Episode Users"),
    # Unactivated
    "unactivated": ("Unactivated / No Meaningful Engagement", "No-Activation / No Meaningful Behavioral History"),
}

# Demographics Team overlays (optional — not every cluster has one)
DEMOGRAPHIC_AUDIENCE = {
    "active_1": "US API Builders",
    "active_3": "US API Builders",
    "active_5": "US API Builders",
    "active_0": "APAC Learners",
    "active_2": "APAC Learners",
    "active_4": "APAC Learners",
}

# Archetypes can apply to multiple clusters; some clusters match more than one.
DEMOGRAPHIC_ARCHETYPES = {
    "at_risk_2": ["DLI Dropout", "Deployment Wall"],
    "at_risk_1": ["Deployment Wall"],
    "at_risk_3": ["Deployment Wall"],
    "Dormant_One_Time_Users": ["Conference Bounce"],
    "Dormant_Former_Builders": ["Former Builders (Recovery Opportunity)"],
}

GHOST_CLUSTERS = {"at_risk_4", "cooling_3"}


def build_cluster_reference() -> pd.DataFrame:
    rows = []
    for cluster_key, (feature_name, technical_name) in PERSONA_NAMES.items():
        archetypes = DEMOGRAPHIC_ARCHETYPES.get(cluster_key, [])
        rows.append(
            {
                "cluster_key": cluster_key,
                "persona_feature_name": feature_name,
                "persona_technical_name": technical_name,
                "demographic_audience": DEMOGRAPHIC_AUDIENCE.get(cluster_key),
                "demographic_archetypes": "; ".join(archetypes) if archetypes else None,
                "is_ghost_cluster": cluster_key in GHOST_CLUSTERS,
            }
        )
    ref = pd.DataFrame(rows)

    def combined_label(row: pd.Series) -> str:
        parts = []
        if pd.notna(row["demographic_audience"]):
            parts.append(row["demographic_audience"])
        if pd.notna(row["demographic_archetypes"]):
            parts.append(row["demographic_archetypes"])
        if row["is_ghost_cluster"]:
            parts.append("Ghost Cluster (unprofileable)")
        parts.append(row["persona_feature_name"])
        return " · ".join(parts)

    ref["cluster_label_combined"] = ref.apply(combined_label, axis=1)
    ref["cluster_label_slide"] = ref["persona_feature_name"].where(
        ref["demographic_audience"].isna(),
        ref["demographic_audience"] + " · " + ref["persona_feature_name"],
    )
    return ref.sort_values(["cluster_key"]).reset_index(drop=True)


cluster_ref = build_cluster_reference()
cluster_ref

,cluster_key,persona_feature_name,persona_technical_name,demographic_audience,demographic_archetypes,is_ghost_cluster,cluster_label_combined,cluster_label_slide
0,Dormant_Former_Builders,Dormant Former Builders,Dormant Historical Builders,NaN,Former Builders (Recovery Opportunity),False,Former Builders (Recovery Opportunity) · Dormant Former Builders,Dormant Former Builders
1,Dormant_Low_Depth,Dormant Low-Depth Visitors,Dormant Low-Depth Historical Users,NaN,NaN,False,Dormant Low-Depth Visitors,Dormant Low-Depth Visitors
2,Dormant_One_Time_Users,Dormant One-Time Users,Dormant Single-Episode Users,NaN,Conference Bounce,False,Conference Bounce · Dormant One-Time Users,Dormant One-Time Users
3,active_0,GenAI Evaluation Cohort,GenAI Evaluators with Education-Led Activity,APAC Learners,NaN,False,APAC Learners · GenAI Evaluation Cohort,APAC Learners · GenAI Evaluation Cohort
4,active_1,Lightweight GenAI Evaluators,Lower-Depth GenAI Evaluators,US API Builders,NaN,False,US API Builders · Lightweight GenAI Evaluators,US API Builders · Lightweight GenAI Evaluators
5,active_2,GenAI Community Participants,GenAI Evaluators with Community-Led Activity,APAC Learners,NaN,False,APAC Learners · GenAI Community Participants,APAC Learners · GenAI Community Participants
6,active_3,High-Intent GenAI Evaluators,Higher-Intent GenAI Evaluators with Stronger Training Activity,US API Builders,NaN,False,US API Builders · High-Intent GenAI Evaluators,US API Builders · High-Intent GenAI Evaluators
7,active_4,Event-Driven GenAI Evaluators,GenAI Evaluators with Webinar-Led Activity,APAC Learners,NaN,False,APAC Learners · Event-Driven GenAI Evaluators,APAC Learners · Event-Driven GenAI Evaluators
8,active_5,Core CUDA Builders,CUDA Builders with Strong API and Download Depth,US API Builders,NaN,False,US API Builders · Core CUDA Builders,US API Builders · Core CUDA Builders
9,active_noise,Active Multi-Path Users,Mixed Active Users with Broad Activity Paths,NaN,NaN,False,Active Multi-Path Users,Active Multi-Path Users


## 3. Load V11 membership

In [3]:
def load_membership() -> pd.DataFrame:
    if DB_PATH.exists():
        con = duckdb.connect(str(DB_PATH), read_only=True)
        try:
            exists = (
                con.execute(
                    """
                    SELECT COUNT(*)
                    FROM information_schema.tables
                    WHERE table_schema = 'main' AND table_name = ?
                    """,
                    [MEMBERSHIP_TABLE],
                ).fetchone()[0]
                > 0
            )
            if exists:
                df = con.execute(
                    f"""
                    SELECT developer_id, stratum, cluster_key
                    FROM {MEMBERSHIP_TABLE}
                    """
                ).fetchdf()
                print(f"Loaded {len(df):,} rows from DuckDB table {MEMBERSHIP_TABLE}")
                return df
        finally:
            con.close()

    if MEMBERSHIP_PARQUET.exists():
        df = pd.read_parquet(MEMBERSHIP_PARQUET, columns=["developer_id", "stratum", "cluster_key"])
        print(f"Loaded {len(df):,} rows from {MEMBERSHIP_PARQUET}")
        return df

    raise FileNotFoundError(
        f"Could not find {MEMBERSHIP_TABLE} in DuckDB or {MEMBERSHIP_PARQUET}"
    )


membership = load_membership()
membership.head()

Loaded 9,381,508 rows from DuckDB table dev_lifecycle_cluster_membership_v11_final


,developer_id,stratum,cluster_key
0,7157208,active,active_noise
1,9951223,active,active_noise
2,7573245,active,active_noise
3,10560864,active,active_2
4,7197215,active,active_noise


## 4. Assign names and validate coverage

In [4]:
named = membership.merge(cluster_ref, on="cluster_key", how="left")

missing_persona = sorted(named.loc[named["persona_feature_name"].isna(), "cluster_key"].unique())
extra_in_registry = sorted(set(PERSONA_NAMES) - set(membership["cluster_key"].unique()))

if missing_persona:
    raise RuntimeError(
        "Membership contains cluster_key values with no persona name: " + ", ".join(missing_persona)
    )

print(f"Developers named: {len(named):,}")
print(f"Distinct cluster_key values in membership: {membership['cluster_key'].nunique()}")
print(f"Registry cluster_key values not present in membership: {extra_in_registry or 'none'}")
print(f"Developers with demographic audience label: {named['demographic_audience'].notna().sum():,}")
print(f"Developers with demographic archetype label: {named['demographic_archetypes'].notna().sum():,}")
print(f"Developers in ghost clusters: {named['is_ghost_cluster'].sum():,}")

named.head(10)

Developers named: 9,381,508
Distinct cluster_key values in membership: 26
Registry cluster_key values not present in membership: none
Developers with demographic audience label: 312,326
Developers with demographic archetype label: 3,400,872
Developers in ghost clusters: 142,992


,developer_id,stratum,cluster_key,persona_feature_name,persona_technical_name,demographic_audience,demographic_archetypes,is_ghost_cluster,cluster_label_combined,cluster_label_slide
0,7157208,active,active_noise,Active Multi-Path Users,Mixed Active Users with Broad Activity Paths,NaN,NaN,False,Active Multi-Path Users,Active Multi-Path Users
1,9951223,active,active_noise,Active Multi-Path Users,Mixed Active Users with Broad Activity Paths,NaN,NaN,False,Active Multi-Path Users,Active Multi-Path Users
2,7573245,active,active_noise,Active Multi-Path Users,Mixed Active Users with Broad Activity Paths,NaN,NaN,False,Active Multi-Path Users,Active Multi-Path Users
3,10560864,active,active_2,GenAI Community Participants,GenAI Evaluators with Community-Led Activity,APAC Learners,NaN,False,APAC Learners · GenAI Community Participants,APAC Learners · GenAI Community Participants
4,7197215,active,active_noise,Active Multi-Path Users,Mixed Active Users with Broad Activity Paths,NaN,NaN,False,Active Multi-Path Users,Active Multi-Path Users
5,3906255,active,active_noise,Active Multi-Path Users,Mixed Active Users with Broad Activity Paths,NaN,NaN,False,Active Multi-Path Users,Active Multi-Path Users
6,775247,active,active_noise,Active Multi-Path Users,Mixed Active Users with Broad Activity Paths,NaN,NaN,False,Active Multi-Path Users,Active Multi-Path Users
7,8915315,active,active_noise,Active Multi-Path Users,Mixed Active Users with Broad Activity Paths,NaN,NaN,False,Active Multi-Path Users,Active Multi-Path Users
8,10266210,active,active_0,GenAI Evaluation Cohort,GenAI Evaluators with Education-Led Activity,APAC Learners,NaN,False,APAC Learners · GenAI Evaluation Cohort,APAC Learners · GenAI Evaluation Cohort
9,8506204,active,active_noise,Active Multi-Path Users,Mixed Active Users with Broad Activity Paths,NaN,NaN,False,Active Multi-Path Users,Active Multi-Path Users


In [5]:
summary = (
    named.groupby(
        [
            "stratum",
            "cluster_key",
            "persona_feature_name",
            "demographic_audience",
            "demographic_archetypes",
            "is_ghost_cluster",
            "cluster_label_combined",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="n_developers")
    .sort_values(["stratum", "n_developers"], ascending=[True, False])
)
summary

,stratum,cluster_key,persona_feature_name,demographic_audience,demographic_archetypes,is_ghost_cluster,cluster_label_combined,n_developers
5,active,active_5,Core CUDA Builders,US API Builders,NaN,False,US API Builders · Core CUDA Builders,155026
6,active,active_noise,Active Multi-Path Users,NaN,NaN,False,Active Multi-Path Users,105723
1,active,active_1,Lightweight GenAI Evaluators,US API Builders,NaN,False,US API Builders · Lightweight GenAI Evaluators,66683
3,active,active_3,High-Intent GenAI Evaluators,US API Builders,NaN,False,US API Builders · High-Intent GenAI Evaluators,29395
2,active,active_2,GenAI Community Participants,APAC Learners,NaN,False,APAC Learners · GenAI Community Participants,24653
4,active,active_4,Event-Driven GenAI Evaluators,APAC Learners,NaN,False,APAC Learners · Event-Driven GenAI Evaluators,18554
0,active,active_0,GenAI Evaluation Cohort,APAC Learners,NaN,False,APAC Learners · GenAI Evaluation Cohort,18015
7,at_risk,at_risk_0,At-Risk Technical Learners,NaN,NaN,False,At-Risk Technical Learners,436295
12,at_risk,at_risk_5,At-Risk Event-Led Learners,NaN,NaN,False,At-Risk Event-Led Learners,365945
9,at_risk,at_risk_2,At-Risk GenAI Training Users,NaN,DLI Dropout; Deployment Wall,False,DLI Dropout; Deployment Wall · At-Risk GenAI Training Users,210377


## 5. Export

In [6]:
OUTPUT_COLS = [
    "developer_id",
    "stratum",
    "cluster_key",
    "persona_feature_name",
    "persona_technical_name",
    "demographic_audience",
    "demographic_archetypes",
    "is_ghost_cluster",
    "cluster_label_slide",
    "cluster_label_combined",
]

dev_out = named[OUTPUT_COLS].copy()
ref_out = cluster_ref.copy()

dev_parquet = OUTPUT_DIR / "dev_v11_cluster_names.parquet"
ref_csv = OUTPUT_DIR / "v11_cluster_name_reference.csv"

dev_out.to_parquet(dev_parquet, index=False)
ref_out.to_csv(ref_csv, index=False)

print(f"Wrote {dev_parquet} ({len(dev_out):,} rows)")
print(f"Wrote {ref_csv} ({len(ref_out)} cluster definitions)")

if WRITE_DUCKDB and DB_PATH.exists():
    con = duckdb.connect(str(DB_PATH))
    try:
        con.register("dev_v11_cluster_names_df", dev_out)
        con.execute(f"DROP TABLE IF EXISTS {DUCKDB_OUTPUT_TABLE}")
        con.execute(
            f"CREATE TABLE {DUCKDB_OUTPUT_TABLE} AS SELECT * FROM dev_v11_cluster_names_df"
        )
        con.unregister("dev_v11_cluster_names_df")
        print(f"Wrote DuckDB table {DUCKDB_OUTPUT_TABLE}")
    finally:
        con.close()
else:
    print("Skipped DuckDB write (disabled or database not found)")

Wrote outputs/v11_cluster_naming/dev_v11_cluster_names.parquet (9,381,508 rows)
Wrote outputs/v11_cluster_naming/v11_cluster_name_reference.csv (26 cluster definitions)
Wrote DuckDB table dev_v11_cluster_names


## 6. Example joins

Use `cluster_label_slide` for stakeholder slides (audience + persona when available). Use `cluster_label_combined` when archetypes / ghost flags should appear.

In [7]:
named.loc[named["demographic_audience"].notna(), [
    "cluster_key", "demographic_audience", "persona_feature_name", "cluster_label_slide"
]].drop_duplicates().sort_values("cluster_key")

,cluster_key,demographic_audience,persona_feature_name,cluster_label_slide
8,active_0,APAC Learners,GenAI Evaluation Cohort,APAC Learners · GenAI Evaluation Cohort
14,active_1,US API Builders,Lightweight GenAI Evaluators,US API Builders · Lightweight GenAI Evaluators
3,active_2,APAC Learners,GenAI Community Participants,APAC Learners · GenAI Community Participants
18,active_3,US API Builders,High-Intent GenAI Evaluators,US API Builders · High-Intent GenAI Evaluators
392,active_4,APAC Learners,Event-Driven GenAI Evaluators,APAC Learners · Event-Driven GenAI Evaluators
29,active_5,US API Builders,Core CUDA Builders,US API Builders · Core CUDA Builders


In [8]:
named.loc[named["demographic_archetypes"].notna(), [
    "cluster_key", "demographic_archetypes", "persona_feature_name", "cluster_label_combined"
]].drop_duplicates().sort_values("cluster_key")

,cluster_key,demographic_archetypes,persona_feature_name,cluster_label_combined
2355426,Dormant_Former_Builders,Former Builders (Recovery Opportunity),Dormant Former Builders,Former Builders (Recovery Opportunity) · Dormant Former Builders
2355455,Dormant_One_Time_Users,Conference Bounce,Dormant One-Time Users,Conference Bounce · Dormant One-Time Users
774906,at_risk_1,Deployment Wall,At-Risk Builder-Learners,Deployment Wall · At-Risk Builder-Learners
774556,at_risk_2,DLI Dropout; Deployment Wall,At-Risk GenAI Training Users,DLI Dropout; Deployment Wall · At-Risk GenAI Training Users
774558,at_risk_3,Deployment Wall,At-Risk GenAI Hands-On Learners,Deployment Wall · At-Risk GenAI Hands-On Learners
